# Notebook 1 of 3: Cell segmentation (CellPose_v4 + Proseg)

**What this notebook does.** It turns one Stereo-seq chip into a table of single cells with gene counts.
It reads the tissue GEF (all transcripts) and the ssDNA image (the nuclei stain), finds each nucleus with the
Cellpose-SAM model, runs Proseg to grow full cell bodies around those nuclei, and saves the result as a `.h5ad`
file. Notebook 2 then does the biology (QC, clustering, cell types, tissue domains).

**Which computer to run this on.** This notebook needs an NVIDIA GPU and the `cellpose` environment.
On this project that is the Windows machine through WSL (Ubuntu). The paths below use the WSL style
(`/mnt/e/...` is the E: drive, `/home/...` is your Linux home). Notebook 2 can run on Mac/Work laptop/Lab computer.

**How to run a notebook, for anyone new to this.**
1. Open a terminal and start JupyterLab in the segmentation environment (see the setup box below).
2. Click a cell to select it, then press **Shift+Enter** to run it and move to the next one.
3. Run the cells **top to bottom, in order**. Do not skip. If a cell is still running you will see `[*]` on its left.
4. Only edit the lines marked `# EDIT`. Everything else can stay as it is.

*Copy paste code block below into new notebook to run it

## Set up once (skip if the environment already exists)

Run these in a terminal on the lab computer (Ubuntu), not in the notebook. This installs the tools this
notebook imports, plus Jupyter itself.

```bash
# a fresh conda environment for GPU segmentation
conda create -n cellpose python=3.10 -y
conda activate cellpose

# cellpose with the SAM model, plus Jupyter and the readers this notebook uses
pip install "cellpose>=3.0" jupyterlab ipykernel tifffile scikit-image h5py zarr anndata scipy numpy pandas matplotlib

# register this env so it shows up by name in JupyterLab
python -m ipykernel install --user --name cellpose --display-name "Python (cellpose)"

# Proseg (the cell-calling step). Needs the Rust toolchain once:
#   curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh
cargo install proseg      # puts the 'proseg' binary in ~/.cargo/bin

# start Jupyter from inside the pipeline folder
cd /path/to/stereoseq_pipeline    # change to where these notebooks live
jupyter lab
```

When JupyterLab opens this notebook, look at the kernel name at the top right. If it does not already say
**Python (cellpose)**, click it (or use Kernel then Change Kernel) and choose **Python (cellpose)**.

To confirm you are on the right kernel, run this in a cell:

```python
import sys; print(sys.executable)
```

The path should end in `envs/cellpose/bin/python`. The final proof is Step 5, where `cellpose GPU available:`
should print `True`.

## Step 1. Settings

Change the sample name and the four paths to match your run, then run the cell. It prints whether each input file
was found. Do not go past this step until both inputs say `True`.

**The two inputs you need** (both come out of the Stereo-seq pipeline for your chip):
- **The tissue GEF**, `SAMPLE.tissue.gef`. It holds every transcript and its position. Usually under
  `.../outs/feature_expression/`.
- **The ssDNA image**, `SAMPLE_ssDNA_regist.tif`. This is the nuclei stain, already registered to the chip.
  Usually under `.../outs/image/`.

**How to find the full path of a file (on the GPU machine, WSL Ubuntu).** WSL sees the Windows `E:` drive as
`/mnt/e`, so a file at `E:\IMKK\...` becomes `/mnt/e/IMKK/...` here. To get an exact path: open the folder in
Windows Explorer and copy it from the address bar, then swap `E:\` for `/mnt/e/` and the backslashes for forward
slashes. Or, in a WSL terminal, `cd` into the folder and run `pwd` and `ls` to read the names.

**What this notebook writes.** Everything goes under the `WORK` folder you set below: the cached nucleus mask, the
transcript table Proseg reads, the Proseg output, and finally `SAMPLE_cpsam_proseg_raw.h5ad`, which is the file
you carry to Notebook 2. Use a fast local disk for `WORK`, not a network or USB drive.

In [ ]:
import time, gzip, csv, itertools, warnings
from pathlib import Path
import numpy as np, pandas as pd
import h5py
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

# EDIT: the sample name.
SAMPLE = "Y40172EA"

# EDIT: Stereo-seq pixel size in microns. 0.5 for standard Stereo-seq. Leave as is unless your chip differs.
PX_UM  = 0.5

# EDIT: where the raw inputs live (WSL sees the E: drive as /mnt/e).
DATA       = Path(f"/mnt/e/IMKK/{SAMPLE}/outs/feature_expression")
TISSUE_GEF = DATA / f"{SAMPLE}.tissue.gef"                                  # all transcripts, from the sequencer
IMG_PATH   = Path(f"/mnt/e/IMKK/{SAMPLE}/outs/image/{SAMPLE}_ssDNA_regist.tif")  # the nuclei stain image

# EDIT: where to write output/results. Use a fast local disk, not the network/USB drive.
WORK        = Path(f"/home/cszla/proseg_runs/{SAMPLE}")

FIG         = WORK / "figures"
CPSAM_DIR   = WORK / "cpsam"
for d in (WORK, FIG, CPSAM_DIR):
    d.mkdir(parents=True, exist_ok=True)

MASK_PATH   = CPSAM_DIR / "mask_cpsam_fullroi.npy"          # the nuclei mask gets cached here
STRIP_CPSAM = WORK / "transcripts_fullroi_cpsam_seed.csv.gz"  # the transcript table Proseg reads

print("tissue gef :", TISSUE_GEF, TISSUE_GEF.exists())
print("ssDNA image:", IMG_PATH,  IMG_PATH.exists())
print("work dir   :", WORK)

## Step 2. Look inside the GEF 

This just lists the data tables stored in the GEF so you can confirm the file opened.
Nothing to change. You can skip it but good to check.

In [ ]:
with h5py.File(TISSUE_GEF, "r") as f:
    def show(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(name, obj.shape, obj.dtype)
    f.visititems(show)

## Step 3. Load the transcripts (full chip)

Look at the picture: it should look like your tissue section, not a blank square or a thin edge. Bright means
more transcripts.

In [ ]:
with h5py.File(TISSUE_GEF, "r") as f:
    expr = f["geneExp/bin1/expression"]
    x = expr["x"][:]
    y = expr["y"][:]
    c = expr["count"][:]

ROI = dict(x0=int(x.min()), x1=int(x.max())+1, #reads the minimum and maximum transcript positions straight from your data and fills in for you
           y0=int(y.min()), y1=int(y.max())+1)
print("ROI:", ROI)

plt.figure(figsize=(8, 8))
plt.hist2d(x, y, weights=c, bins=800, cmap="magma")     # bins=800 is only the picture resolution, not the data
plt.gca().invert_yaxis(); plt.gca().set_aspect("equal")
plt.title(f"{SAMPLE} whole-sample transcript density"); plt.colorbar()
plt.show()

## Step 4. Load the nuclei image

This loads the ssDNA image (the nuclei stain),and stretches the contrast so the
nuclei are easy to see. **Look at the preview:** you should see grey tissue with visible nuclei. If it is
all black or all white, adjust the two percentiles on the `lo, hi` line (widen towards 0 and 100 to keep more,
tighten towards the middle for more contrast).

In [ ]:
import tifffile
from skimage.exposure import rescale_intensity

img = tifffile.imread(str(IMG_PATH))
print("full image:", img.shape, img.dtype)

crop = img[ROI["y0"]:ROI["y1"], ROI["x0"]:ROI["x1"]]
if crop.ndim == 3:
    crop = crop[..., 0]          # keep one channel if the image is RGB
crop = crop.astype(np.float32)

vals   = crop[crop > 0]
lo, hi = np.percentile(vals, [1, 99.8])       # EDIT if the preview is too dark or too bright (e.g. [2, 99.5])
ssdna  = rescale_intensity(crop, in_range=(lo, hi), out_range=(0, 1)).astype(np.float32)
print("cropped ssDNA:", ssdna.shape, "min/max:", ssdna.min(), ssdna.max())

plt.figure(figsize=(8, 8))
plt.imshow(ssdna[::10, ::10], cmap="gray")    # [::10] shows every 10th pixel, just a fast preview
plt.title(f"{SAMPLE} ssDNA ROI"); plt.axis("off")
plt.show()

## Step 5. Test the nucleus finder on one small window

Before segmenting the whole chip (which is slow), test the model on one small square so you can see if it
finds nuclei correctly. **Look at the red outlines:** each real nucleus should get one outline, not merged
blobs and not many pieces per nucleus. `cellpose GPU available: True` should print. If it says `False`, the
GPU is not being used and the full run will be very slow.

To move the test window, change `wy, wx` (the top-left corner in pixels). `ws` is the window size.

In [ ]:
from cellpose import models, core
from skimage.segmentation import find_boundaries

print("cellpose GPU available:", core.use_gpu())          # want True
model = models.CellposeModel(gpu=core.use_gpu(), pretrained_model="cpsam")

wy, wx, ws = 4500, 4500, 1200      # EDIT: window top-left (wy, wx) and size (ws), in pixels
test_img = ssdna[wy:wy+ws, wx:wx+ws]

test_masks, flows, styles = model.eval(
    test_img, channels=[0, 0], diameter=None,     # diameter=None lets the model estimate nucleus size
    normalize=True, tile_overlap=0.1, batch_size=8,
)
test_masks = test_masks.astype(np.uint32)

ov = np.dstack([test_img]*3); ov = (ov*255).clip(0,255).astype(np.uint8)
ov[find_boundaries(test_masks, mode="thick")] = [255, 0, 0]
plt.figure(figsize=(9, 9)); plt.imshow(ov)
plt.title(f"CPSAM test | nuclei found = {int(test_masks.max())}"); plt.axis("off"); plt.show()

## Step 6. Segment every nucleus on the chip (will take some time to run)

This runs the model on the whole chip. The result is cached
to disk, so if you re-run this cell later it loads the saved mask in seconds instead of recomputing. To force
a fresh run, delete the file printed as `MASK_PATH` and run again.

In [ ]:
if MASK_PATH.exists():
    print("loading existing mask:", MASK_PATH)
    masks = np.load(MASK_PATH)
else:
    t0 = time.time()
    masks, flows, styles = model.eval(
        ssdna, channels=[0, 0], diameter=None,
        normalize=True, tile_overlap=0.1, batch_size=8,
    )
    masks = masks.astype(np.uint32)
    np.save(MASK_PATH, masks)
    print("saved:", MASK_PATH, "| CPSAM time:", int(time.time()-t0), "s")

print("mask shape:", masks.shape, "| number of nuclei:", int(masks.max()))
assert masks.shape == ssdna.shape

## Step 7. Check the segmentation in a few places

This shows the outlines over four windows across the tissue so you can confirm the model did well in
different regions, not just the test spot. **Look:** outlines should hug real nuclei everywhere.
The figure is also saved into the `figures` folder for your records. Edit the `windows` list to inspect
other locations (each entry is `(y, x, size)` in pixels).

In [ ]:
windows = [(2500,2500,1200), (4500,4500,1200), (6500,8000,1200), (8500,11000,1200)]   # EDIT: (y, x, size)
fig, axes = plt.subplots(1, len(windows), figsize=(6*len(windows), 6))
for ax, (wy, wx, ws) in zip(axes, windows):
    imgw, maskw = ssdna[wy:wy+ws, wx:wx+ws], masks[wy:wy+ws, wx:wx+ws]
    ov = np.dstack([imgw]*3); ov = (ov*255).clip(0,255).astype(np.uint8)
    ov[find_boundaries(maskw, mode="thick")] = [255, 0, 0]
    ax.imshow(ov); ax.set_title(f"y={wy} x={wx} nuclei={len(np.unique(maskw))-1}"); ax.axis("off")
plt.tight_layout()
plt.savefig(FIG / "cpsam_outline_qc_windows.png", dpi=250, bbox_inches="tight")
plt.show()

## Step 8. Build the transcript table that Proseg reads

Proseg needs one row per transcript with its position, its gene, and which nucleus (if any) it sits inside.
This cell walks through the GEF gene by gene, keeps transcripts inside the ROI, and looks up the nucleus label
under each one. It writes a compressed CSV. This can take a few minutes and prints progress as it goes.

It will not overwrite an existing file. To rebuild, delete the file printed as the output and run again.

In [ ]:
ROW_CHUNK = 200_000       # how many rows to read at a time. Lower it if you run out of memory.
def decode_gene_name(v):
    return v.decode("utf-8", "ignore").rstrip("\x00").strip()

x0, x1, y0, y1 = ROI["x0"], ROI["x1"], ROI["y0"], ROI["y1"]

if STRIP_CPSAM.exists():
    print("exists -> delete this file to rebuild:", STRIP_CPSAM)
else:
    t0 = time.time(); tid = kept = in_mask = 0
    with h5py.File(TISSUE_GEF, "r") as f, gzip.open(STRIP_CPSAM, "wt", newline="") as fout:
        expr  = f["geneExp/bin1/expression"]
        genes = f["geneExp/bin1/gene"][:]
        w = csv.writer(fout)
        w.writerow(["transcript_id","x_location","y_location","z_location",
                    "gene","cell_id","overlaps_nucleus"])
        for gi, g in enumerate(genes):
            gname = decode_gene_name(g["geneName"])
            off, cnt = int(g["offset"]), int(g["count"])
            if cnt == 0:
                continue
            for start in range(off, off+cnt, ROW_CHUNK):
                stop = min(start+ROW_CHUNK, off+cnt)
                rec = expr[start:stop]
                gx = rec["x"].astype(np.int32)
                gy = rec["y"].astype(np.int32)
                gc = rec["count"].astype(np.int32)
                m = (gx>=x0)&(gx<x1)&(gy>=y0)&(gy<y1)
                if not m.any():
                    continue
                xr = np.repeat(gx[m], gc[m])            # one row per transcript (expand the count column)
                yr = np.repeat(gy[m], gc[m])
                if len(xr) == 0:
                    continue
                li = masks[yr-y0, xr-x0].astype(np.int64)   # nucleus label under each transcript (0 = none)
                w.writerows(zip(range(tid, tid+len(xr)), xr.tolist(), yr.tolist(),
                                itertools.repeat(0), itertools.repeat(gname),
                                li.tolist(), (li>0).astype(np.uint8).tolist()))
                tid += len(xr); kept += len(xr); in_mask += int((li>0).sum())
            if gi % 500 == 0:
                print(f"gene {gi:,}/{len(genes):,}  kept {kept:,}  in-nucleus {in_mask:,}  {time.time()-t0:.0f}s")
    print(f"done: {kept:,} transcripts, {in_mask:,} inside a nucleus, {time.time()-t0:.0f}s")
    print("output:", STRIP_CPSAM)

## Step 9. Run Proseg (in the terminal, not in the notebook)

> **Run this step in a terminal window, not in a notebook cell.** Proseg is a separate program. It prints a lot of
> progress and can take 30 to 60 minutes, which would tie up the notebook. Open a terminal, paste the command
> below, edit the one path, and let it run.

Proseg grows full cell bodies out from the nuclei using the transcripts from Step 8.

```bash
cd /home/cszla/proseg_runs/Y40172EA          # EDIT: your WORK folder from Step 1

proseg transcripts_fullroi_cpsam_seed.csv.gz \
    --gene-column gene \
    --x-column x_location --y-column y_location --z-column z_location \
    --cell-id-column cell_id --cell-id-unassigned 0 \
    --voxel-layers 2 \
    --output-dir proseg_cpsam_voxel_2.0_full
```

**Input it reads:** `transcripts_fullroi_cpsam_seed.csv.gz` (written by Step 8, in your `WORK` folder).
**Output it writes:** a folder `proseg_cpsam_voxel_2.0_full/` containing `proseg-counts.mtx.gz`,
`proseg-cell-metadata.csv.gz`, and `proseg-output.zarr`. Step 10 reads those three, so keep Proseg's default
output names.

When Proseg finishes and prints its summary, come back to the notebook and run Step 10.

Notes:
- Flag names can differ slightly between Proseg versions. Run `proseg --help` once and match the flags to your
  version if anything is rejected.
- The next cell (optional) can run Proseg from inside the notebook instead, but the terminal is the recommended
  way.

In [ ]:
# OPTIONAL: run Proseg from inside the notebook instead of the terminal.
# Uncomment (remove the leading #) to use it. It blocks the notebook until Proseg finishes.

# import subprocess
# OUT_DIR = WORK / "proseg_cpsam_voxel_2.0_full"
# cmd = [
#     "proseg", str(STRIP_CPSAM),
#     "--gene-column", "gene",
#     "--x-column", "x_location", "--y-column", "y_location", "--z-column", "z_location",
#     "--cell-id-column", "cell_id", "--cell-id-unassigned", "0",
#     "--voxel-layers", "2",
#     "--output-dir", str(OUT_DIR),
# ]
# print(" ".join(cmd))
# subprocess.run(cmd, cwd=str(WORK), check=True)

## Step 10. Load the Proseg result and save the h5ad

This reads Proseg's cell-by-gene counts, cell centroids, and volumes, packs them into an AnnData object, and
saves it as a single `.h5ad` file. That file is the input to Notebook 2. When it prints the object and the file
size, this notebook is done.

In [ ]:
import scipy.io, zarr, anndata as ad

OUT  = WORK / "proseg_cpsam_voxel_2.0_full"                 # EDIT if you used a different --output-dir
H5AD = WORK / f"{SAMPLE}_cpsam_proseg_raw.h5ad"

with gzip.open(OUT / "proseg-counts.mtx.gz", "rt") as fh:
    X = scipy.io.mmread(fh).tocsr().astype(np.float32)      # cells x genes count matrix

obs = pd.read_csv(OUT / "proseg-cell-metadata.csv.gz")      # one row per cell (centroid, volume, ...)
obs.index = obs["cell"].astype(str).values

genes = np.asarray(
    zarr.open(str(OUT / "proseg-output.zarr/tables/table"), mode="r")["var"]["_index"][:],
    dtype=str,
)

adata = ad.AnnData(X=X, obs=obs, var=pd.DataFrame(index=genes))
adata.var_names_make_unique()
adata.obsm["spatial"]      = obs[["centroid_x", "centroid_y"]].to_numpy(np.float32)
adata.layers["counts"]     = adata.X.copy()                 # keep a clean copy of the raw counts
adata.obs["total_counts"]  = np.asarray(adata.X.sum(1)).ravel()

print(adata)
adata.write_h5ad(H5AD, compression="gzip")
print("saved:", H5AD, "| size MB:", round(H5AD.stat().st_size / 1e6, 1))
print("\nNext: copy this .h5ad to the Mac/Computer you are using for downstream analysis and open Notebook 2.")